# Mounter

> Directory mounting for static file serving in FastHTML applications.

In [ ]:
#| default_exp serving.mounter

In [ ]:
#| export
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional
from urllib.parse import quote

## DirectoryMounter

Mounts directories for static file serving and generates URLs for files.

In [ ]:
#| export
class DirectoryMounter:
    """Mounts directories for static file serving."""
    
    def __init__(
        self,
        prefix: str = "media",  # URL prefix for mounted files
    ):
        """Initialize with URL prefix."""
        self.prefix = prefix.strip("/")
        self._mounts: Dict[str, str] = {}  # mount_id -> directory path
        self._path_to_mount: Dict[str, str] = {}  # normalized path -> mount_id
    
    def _generate_mount_id(
        self,
        directory: str  # Directory path
    ) -> str:  # Short unique ID
        """Generate a short unique ID for a directory."""
        # Use hash of path for consistent IDs
        path_hash = hashlib.md5(directory.encode()).hexdigest()[:8]
        return path_hash
    
    def _normalize_path(
        self,
        path: str  # Path to normalize
    ) -> str:  # Normalized path
        """Normalize a path for consistent comparison."""
        return str(Path(path).resolve())
    
    def mount(
        self,
        directory: str  # Directory to mount
    ) -> str:  # Mount ID
        """Mount a directory and return its mount ID."""
        normalized = self._normalize_path(directory)
        
        # Check if already mounted
        if normalized in self._path_to_mount:
            return self._path_to_mount[normalized]
        
        # Generate mount ID and register
        mount_id = self._generate_mount_id(normalized)
        self._mounts[mount_id] = normalized
        self._path_to_mount[normalized] = mount_id
        
        return mount_id
    
    def mount_directories(
        self,
        directories: List[str]  # Directories to mount
    ) -> Dict[str, str]:  # Map of directory -> mount_id
        """Mount multiple directories."""
        result = {}
        for directory in directories:
            result[directory] = self.mount(directory)
        return result
    
    def get_url(
        self,
        file_path: str  # Full file path
    ) -> Optional[str]:  # URL or None if not in mounted directory
        """Get URL for a file if it's in a mounted directory."""
        normalized = self._normalize_path(file_path)
        file_path_obj = Path(normalized)
        
        # Find which mount this file belongs to
        for mount_id, mount_path in self._mounts.items():
            mount_path_obj = Path(mount_path)
            try:
                # Get relative path from mount point
                relative = file_path_obj.relative_to(mount_path_obj)
                # URL-encode the relative path
                encoded_path = "/".join(quote(part, safe="") for part in relative.parts)
                return f"/{self.prefix}/{mount_id}/{encoded_path}"
            except ValueError:
                continue
        
        return None
    
    def get_file_path(
        self,
        mount_id: str,  # Mount ID
        relative_path: str  # Relative path within mount
    ) -> Optional[str]:  # Full file path or None
        """Get full file path from mount ID and relative path."""
        if mount_id not in self._mounts:
            return None
        
        mount_path = self._mounts[mount_id]
        full_path = Path(mount_path) / relative_path
        
        # Security: ensure path doesn't escape mount directory
        try:
            full_path.resolve().relative_to(Path(mount_path).resolve())
        except ValueError:
            return None  # Path traversal attempt
        
        return str(full_path)
    
    def is_mounted(
        self,
        directory: str  # Directory to check
    ) -> bool:  # True if mounted
        """Check if a directory is mounted."""
        normalized = self._normalize_path(directory)
        return normalized in self._path_to_mount
    
    def get_mounted_directories(
        self
    ) -> List[str]:  # List of mounted directory paths
        """Get list of mounted directories."""
        return list(self._mounts.values())
    
    def unmount(
        self,
        directory: str  # Directory to unmount
    ) -> bool:  # True if was mounted
        """Unmount a directory."""
        normalized = self._normalize_path(directory)
        if normalized not in self._path_to_mount:
            return False
        
        mount_id = self._path_to_mount[normalized]
        del self._mounts[mount_id]
        del self._path_to_mount[normalized]
        return True
    
    def unmount_all(
        self
    ) -> int:  # Number of mounts removed
        """Remove all mounts."""
        count = len(self._mounts)
        self._mounts.clear()
        self._path_to_mount.clear()
        return count
    
    def create_url_getter(
        self
    ) -> callable:  # Function that converts path to URL
        """Create a URL getter function for use with gallery components."""
        def get_url(path: str) -> Optional[str]:
            return self.get_url(path)
        return get_url

In [ ]:
import tempfile
import os

# Test DirectoryMounter
with tempfile.TemporaryDirectory() as tmpdir:
    # Create test structure
    subdir = Path(tmpdir) / "photos"
    subdir.mkdir()
    test_file = subdir / "image.jpg"
    test_file.write_text("test")
    
    mounter = DirectoryMounter(prefix="media")
    
    # Test mount
    mount_id = mounter.mount(str(subdir))
    assert len(mount_id) == 8  # Hash-based ID
    assert mounter.is_mounted(str(subdir))
    
    # Test get_url
    url = mounter.get_url(str(test_file))
    assert url is not None
    assert url.startswith("/media/")
    assert mount_id in url
    assert "image.jpg" in url
    
    # Test file not in mount
    other_path = Path(tmpdir) / "other.txt"
    other_path.write_text("other")
    assert mounter.get_url(str(other_path)) is None
    
    # Test get_file_path
    resolved = mounter.get_file_path(mount_id, "image.jpg")
    assert resolved is not None
    assert Path(resolved).exists()
    
    # Test path traversal protection
    bad_path = mounter.get_file_path(mount_id, "../../../etc/passwd")
    assert bad_path is None
    
    # Test mounted directories
    dirs = mounter.get_mounted_directories()
    assert len(dirs) == 1
    
    # Test unmount
    assert mounter.unmount(str(subdir)) == True
    assert not mounter.is_mounted(str(subdir))
    assert mounter.get_url(str(test_file)) is None

print("DirectoryMounter tests passed!")

DirectoryMounter tests passed!


In [ ]:
# Test URL getter
with tempfile.TemporaryDirectory() as tmpdir:
    subdir = Path(tmpdir) / "media"
    subdir.mkdir()
    (subdir / "video.mp4").write_text("test")
    
    mounter = DirectoryMounter()
    mounter.mount(str(subdir))
    
    # Create getter
    get_url = mounter.create_url_getter()
    
    # Test getter
    url = get_url(str(subdir / "video.mp4"))
    assert url is not None
    assert "video.mp4" in url

print("URL getter tests passed!")

URL getter tests passed!


In [ ]:
# Test multiple mounts
with tempfile.TemporaryDirectory() as tmpdir:
    dir1 = Path(tmpdir) / "dir1"
    dir2 = Path(tmpdir) / "dir2"
    dir1.mkdir()
    dir2.mkdir()
    (dir1 / "file1.txt").write_text("1")
    (dir2 / "file2.txt").write_text("2")
    
    mounter = DirectoryMounter()
    mounts = mounter.mount_directories([str(dir1), str(dir2)])
    
    assert len(mounts) == 2
    assert len(mounter.get_mounted_directories()) == 2
    
    # Both files should have URLs
    url1 = mounter.get_url(str(dir1 / "file1.txt"))
    url2 = mounter.get_url(str(dir2 / "file2.txt"))
    assert url1 is not None
    assert url2 is not None
    assert url1 != url2
    
    # Test unmount_all
    count = mounter.unmount_all()
    assert count == 2
    assert len(mounter.get_mounted_directories()) == 0

print("Multiple mounts tests passed!")

Multiple mounts tests passed!


## Route Helper

Helper function to create the static file serving route.

In [ ]:
#| export
def create_media_serve_route(
    mounter: DirectoryMounter,  # The mounter instance
) -> callable:  # Route handler function
    """
    Create a route handler for serving media files.
    
    Usage:
    ```python
    from starlette.responses import FileResponse
    
    mounter = DirectoryMounter(prefix="media")
    serve_media = create_media_serve_route(mounter)
    
    @app.get("/media/{mount_id}/{path:path}")
    def media_route(mount_id: str, path: str):
        return serve_media(mount_id, path)
    ```
    """
    def serve_media(
        mount_id: str,  # Mount identifier
        path: str       # Relative file path
    ) -> Any:  # FileResponse or error
        """Serve a media file."""
        from starlette.responses import FileResponse
        from fasthtml.common import Response
        
        file_path = mounter.get_file_path(mount_id, path)
        
        if file_path is None:
            return Response("Not found", status_code=404)
        
        if not Path(file_path).exists():
            return Response("Not found", status_code=404)
        
        return FileResponse(file_path)
    
    return serve_media

In [ ]:
# Test route creation (basic test without actual HTTP)
with tempfile.TemporaryDirectory() as tmpdir:
    test_file = Path(tmpdir) / "test.txt"
    test_file.write_text("hello")
    
    mounter = DirectoryMounter()
    mount_id = mounter.mount(tmpdir)
    
    serve = create_media_serve_route(mounter)
    
    # Test serving existing file
    response = serve(mount_id, "test.txt")
    # Should be FileResponse
    assert hasattr(response, 'path') or hasattr(response, 'status_code')
    
    # Test non-existent file
    response = serve(mount_id, "nonexistent.txt")
    assert response.status_code == 404
    
    # Test invalid mount
    response = serve("invalid", "test.txt")
    assert response.status_code == 404

print("Route helper tests passed!")

Route helper tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()